# Café Data Analysis
On this notebook we will be analysing the data collected by a Café over a period of a year.


## Table of Contents
- [Part 1 - EDA](#part-1)
  - [Step 1 - Setup](#step-1)
    - [Step 1.1 - Libraries and modules](#step-1-1)
  - [Step 2 - Basic Inspection](#step-2)
    - [Step 2.1 - Loading the data](#step-2-1)
    - [Step 2.2 - Dataset size](#step-2-2)
    - [Step 2.3 - Column types](#step-2-3)
    - [Step 2.4 - Unique values](#step-2-4)
    - [Step 2.5 - Summary statistics](#step-2-5)
  - [Step 3 - Data Cleaning and Preparation](#step-3)
    - [Step 3.1 - Missing values](#step-3-1)
    - [Step 3.2 - Text formatting and type conversion](#step-3-2)
    - [Step 3.3 - Duplicate detection](#step-3-3)
    - [Step 3.4 - Fixing item names](#step-3-4)
    - [Step 3.5 - Fixing categories](#step-3-5)
    - [Step 3.6 - Aggregating by order](#step-3-6)
    - [Step 3.7 - Data limitations](#step-3-7)
  - [Step 4 - Descriptive Statistics](#step-4)
    - [Step 4.1 - Headline figures](#step-4-1)
    - [Step 4.2 - Measures of center and spread](#step-4-2)
    - [Step 4.3 - Distribution shapes](#step-4-3)
    - [Step 4.4 - Most sold products](#step-4-4)
    - [Step 4.5 - Most sold categories](#step-4-5)
    - [Step 4.6 - The top 3 categories](#step-4-6)
    - [Step 4.7 - Top products in the top 3 categories](#step-4-7)
  - [Step 5 - Time Series Analysis and Sales Trend Investigation](#step-5)
    - [Step 5.1 - Trend isolation](#step-5-1)
    - [Step 5.2 - Structural decomposition](#step-5-2)
    - [Step 5.3 - Seasonality and recurring cycles](#step-5-3)
    - [Step 5.4 - Detrending and stationarity](#step-5-4)
    - [Step 5.5 - Key takeaways](#step-5-5)
  - [Step 6 - Category Performance and Relationship Discovery](#step-6)
    - [Step 6.1 - Category performance](#step-6-1)
    - [Step 6.2 - Concentration of sales (Pareto)](#step-6-2)
    - [Step 6.3 - Category mix through the day and the week](#step-6-3)
    - [Step 6.4 - Most sold products per month and season](#step-6-4)
    - [Step 6.5 - Relationships](#step-6-5)
    - [Step 6.6 - Anomalies](#step-6-6)
    - [Step 6.7 - Key takeaways](#step-6-7)
  - [Step 7 - Offensive Data Preparation & Feature Engineering](#step-7)
    - [Step 7.1 - Promotion and base product](#step-7-1)
    - [Step 7.2 - Time, drink and bill features](#step-7-2)
    - [Step 7.3 - Order-level features](#step-7-3)
    - [Step 7.4 - Saving the cleaned data](#step-7-4)
- [Part 2 - Market Basket Analysis](#part-2)
  - [Step 1 - Dataset Selection and Feature Engineering for MBA](#p2-step-1)
  - [Step 2 - Algorithm Selection and Metrics](#p2-step-2)
  - [Step 3 - Threshold Calibration and Overall Rules](#p2-step-3)
  - [Step 4 - Combo Deals per Daypart](#p2-step-4)
  - [Step 5 - Exporting the rules](#p2-step-5)
  - [Step 6 - Recommendations and Limitations](#p2-step-6)
- [Part 3 - Red Bull Promotions, Traffic & Price Controls, and Deal-Buying Orders](#part-3)
  - [Step 1 - Setup & Analytical Questions](#p3-step-1)
    - [Step 1.1 - Questions and hypotheses](#p3-step-1-1)
    - [Step 1.2 - Libraries and cleaned line data](#p3-step-1-2)
  - [Step 2 - Red Bull Revenue After Controlling for Weekends and the January 2011 Price Increase](#p3-step-2)
    - [Step 2.1 - Building the daily Red Bull and café traffic dataset from lines](#p3-step-2-1)
    - [Step 2.2 - Promo vs. non-promo revenue across weekend traffic and January 2011 pricing](#p3-step-2-2)
    - [Step 2.3 - Regression check on Red Bull promotion lift](#p3-step-2-3)
  - [Step 3 - Which Customer Orders Are Most Likely to Buy a Deal?](#p3-step-3)
    - [Step 3.1 - Aggregating bill-level orders from cafe_clean_lines.csv](#p3-step-3-1)
    - [Step 3.2 - Deal attachment by order composition, basket size, daypart, and order value](#p3-step-3-2)
    - [Step 3.3 - Modeling deal likelihood across customer orders](#p3-step-3-3)
  - [Step 4 - Key Takeaways & Limitations](#p3-step-4)
    - [Step 4.1 - Summary of findings](#p3-step-4-1)
    - [Step 4.2 - Data limitations](#p3-step-4-2)

<a id="part-1"></a>
## Part 1 - EDA

Note: Claude Code was used in this project

<a id="step-1"></a>
### Step 1 - Setup
Our task is to explore the data, identify interesting patterns and develop insights to help the business improve its performance.

<a id="step-1-1"></a>
#### Step 1.1 - Libraries and modules
The calculations for each step live in separate Python files in the `scripts/` folder (`cleaning.py`, `descriptive.py`, `time_series.py`, `categories.py`, `features.py`), so this notebook only shows the results and what they mean. The libraries (pandas, matplotlib, seaborn, ...) are imported inside those files, and `plot_style.py` sets the same look for every plot. The raw and cleaned data are in `data/` and the files we export (e.g. the market basket rules) go to `outputs/`. The light gray grid is on by default and kept for bar charts and scatter plots, where it helps to read the values; for time series we turn it off with `with ps.grid(False):` so the line itself stands out.

In [ ]:
import sys
sys.path.append("scripts")   # our own modules live in the scripts/ folder

# Our own modules: the calculations for each step
import cleaning as cl       # Steps 2-3
import descriptive as ds    # Step 4
import time_series as ts    # Step 5
import categories as cat    # Step 6
import features as fe       # Step 7
import plot_style as ps     # grid on/off: ps.set_grid(False) for all plots, or `with ps.grid(False):` for one
import apriori_market_basket_analysis_daypart_overview as mba   # Part 2

<a id="step-2"></a>
### Step 2 - Basic Inspection
A first look at the raw data: its shape, column types, unique values and summary statistics.

<a id="step-2-1"></a>
#### Step 2.1 - Loading the data

In [ ]:
# Import and inspect (the file is not UTF-8, so it is read as latin-1, see cleaning.load_raw)
df_original = cl.load_raw("data/cafe.csv")
df_original.head()

<a id="step-2-2"></a>
#### Step 2.2 - Dataset size

In [ ]:
df_original.shape

The dataset contains 145_830 rows and 10 columns.

<a id="step-2-3"></a>
#### Step 2.3 - Column types

In [ ]:
df_original.info()

Here we noticed that the rate, tax, discount and total were strings instead of floats (because it's money), and that the date and time are also strings instead of datetimes.

Another thing is that there is a column called category which can be used to categorize the data.

<a id="step-2-4"></a>
#### Step 2.4 - Unique values

In [ ]:
df_original.nunique()

This shows us the 365 dates (days) and 9 different categories.

<a id="step-2-5"></a>
#### Step 2.5 - Summary statistics

In [ ]:
df_original.describe()

Describe only shows Quantity, since like mentioned before, columns that should be floats, aren't. This step will be repeated later.

These columns will be transformed into the right type in the data cleaning step.

<a id="step-3"></a>
### Step 3 - Data Cleaning and Preparation
Before analysing the data we clean it, step by step:
- **3.1** Check for missing values
- **3.2** Fix the text formatting and convert columns to the right types
- **3.3** Detect duplicate rows
- **3.4** Fix broken item names and different names for the same product
- **3.5** Fix products filed in the wrong category
- **3.6** Merge repeated items in the same order and build one row per order
- **3.7** List the limitations of the data

From this point onwards we work on a copy of the dataset (`df_copy`), so the raw data (`df_original`) stays untouched.

In [ ]:
# Work on a copy, so the raw data stays untouched
df_copy = df_original.copy()

<a id="step-3-1"></a>
#### Step 3.1 - Missing values

Next we decided to see if there are any missing values and think on how to handle them

In [ ]:
df_copy.isnull().sum()

The dataset doesn't contain any missing values, this step can be safely skipped.

<a id="step-3-2"></a>
#### Step 3.2 - Text formatting and type conversion

Two formatting problems:
- **Extra spaces**: the column `Bill Number ` and the category `LIQUOR ` have a trailing space. Because of that, pandas sees "LIQUOR " and "LIQUOR" as **two different categories**. We remove leading and trailing spaces from the column names and text columns.
- **Wrong types**: the money columns use `,` instead of `.` as decimal separator, so Python reads them as text. We convert them to numbers, and the date to a real date.

In [ ]:
df_copy = cl.strip_text(df_copy)
df_copy = cl.convert_types(df_copy)
print("Categories:", sorted(df_copy["Category"].unique()))
df_copy.dtypes

In [ ]:
df_copy.describe()

Something interesting that popped to us immediately is how the min for Rate (how much something costs before tax) is 0.01 and the max is 2100, this seems highly suspicious and flagged for later exploration, but for now let's continue exploring. *(Checked in Step 6: the 0.01 lines are a free Mother's Day item.)*

<a id="step-3-3"></a>
#### Step 3.3 - Duplicate detection
Here we check for duplicate rows and decide how to handle them.

In [ ]:
print(f"Number of duplicate rows: {df_copy.duplicated().sum()}")
cl.duplicate_rows(df_copy)

<a id="step-3-4"></a>
#### Step 3.4 - Fixing item names
While looking at the most sold products we noticed some strange item names, like `RED SANGRIA (GLS)á áááááááááá`. So we went through all 580 product names to find names that are broken, or different names for the same product. We:
- searched for characters that are not normal letters, numbers or punctuation
- compared the names after removing spaces and punctuation, and with a similarity score (`difflib`) to find almost-identical names
- for every suspicious pair, compared the **price** and the **dates** when each name was used: if one name stops being used when the other starts, and the price matches, it's most likely the same product

What we found and fix here:

| Problem | Example | Why we fix it |
|---|---|---|
| **Junk characters** at the end of 4 names | `RED SANGRIA (GLS)á ááááá`, `WHITE SANGRIA (CARAFE)áááá` | Encoding garbage from the till system, not part of the name |
| **Double spaces** in 13 names | `N R G  HOOKAH` (2,314 units) vs `N R G HOOKAH` (15 units) | The same hookah at the same price (450) was counted as 2 products |
| **Spelling typo** | `MORROCAN SOY KEBABS` (Jun-Sep 2010) vs `MOROCCAN SOY KEBABS` (Jan-Mar 2011) | Never used at the same time, so the same dish was renamed when the menu changed |
| **Same product, two names** | `STELLA ARTOIS` (4 units) vs `STELLA ARTOIS (GLS)` (30 units) | Same beer, same price (225) |

Names that only look alike were **not** merged because they are different products with different prices, e.g. glass vs bottle (`(GLS)`/`(BTL)`), sizes (`JR.`/`SR.CHL AVALANCHE`), add-ons (`ADD CHICKEN SAUSAGE` vs `CHICKEN SAUSAGE`) and promotions (`1+1 VLN CAB SAUV (GLS)`, handled in Step 7).

In [ ]:
names_before = df_copy["Item Desc"].nunique()
df_copy = cl.fix_item_names(df_copy)
print(f"Unique products: {names_before} -> {df_copy['Item Desc'].nunique()}")

fixed = ["RED SANGRIA (GLS)", "RED SANGRIA (CARAFE)", "WHITE SANGRIA (GLS)", "WHITE SANGRIA (CARAFE)",
         "N R G HOOKAH", "MOROCCAN SOY KEBABS", "STELLA ARTOIS (GLS)"]
df_copy[df_copy["Item Desc"].isin(fixed)].groupby("Item Desc")["Quantity"].sum().to_frame("Units sold")

<a id="step-3-5"></a>
#### Step 3.5 - Fixing categories
When we looked at the best sellers per category (Step 4), some products were clearly in the wrong category, e.g. the **Great Lakes Shake was the top seller of FOOD**. So we checked the product list of every category, and every product that was booked under more than one category. What we found:

| Problem | Example | How we fix it |
|---|---|---|
| **Typo** in a category name | `LIQUOR & TPBACCO` | Renamed to `LIQUOR & TOBACCO` |
| **Same product in several categories** (22 products) | `ORANGE ARRABIATA`: 493x FOOD, 2x MISC | Use the category the product is booked under most often. Almost all of these MISC bookings are from **December 2010** |
| **Milkshakes filed as FOOD** (14 shakes, 16,170 units) | `GREAT LAKES SHAKE`, `OREO COOKIE SHAKE` | Moved to BEVERAGE, where the other shakes (`M & M SHAKE`, `STRAWBERRY LITE SHAKE`) already are |
| **Wines filed elsewhere** | Sangria and Zinzi under LIQUOR, `LATE HARVEST SULA CHENIN` under TOBACCO | Moved to WINES |
| **New December products booked as MISC** | 5 new sheeshas, `KF DRAFT MUG`, `GIN (SM)` | Moved to their real category. This was the cause of the **MISC spike in December** |
| **Hookah flavours and drinks always booked as MISC** | `ROSE FLAVORS SINGLE`, `PEACH BULL` | Moved to TOBACCO / BEVERAGE |

What stays in MISC are real extras: add-ons (`ADD ON S`, `ADD POTATO WEDGES`), party charges and a few products we can't place (`GUAVA`, `PLAIN JANE`). The combined `LIQUOR & TOBACCO` category is kept as it is, until we decide with the team how to handle it.

Moving the shakes changes the category totals a lot (BEVERAGE becomes the biggest category in units), so all the following steps use the corrected categories.

In [ ]:
before = df_copy.copy()
df_copy = cl.fix_categories(df_copy)
print("Products in more than one category:", (df_copy.groupby("Item Desc")["Category"].nunique() > 1).sum())
cl.category_changes(before, df_copy)

<a id="step-3-6"></a>
#### Step 3.6 - Aggregating by order
We assumed that sometimes the waiter entered the same item several times in one bill (e.g. 3 x "panini x1" instead of "panini x3"). Here we merge those rows directly in `df_copy`, summing quantity and money, so each item appears once per order.

We also build `df_orders`: one row per order (bill), with the items bought, the total quantity and the money columns summed.

In [ ]:
# Merge repeated rows of the same item in the same order into one row
rows_before = len(df_copy)
df_copy = cl.merge_repeated_items(df_copy)
print(f"Rows: {rows_before} -> {len(df_copy)} ({rows_before - len(df_copy)} rows merged)")

# One row per order (bill)
df_orders = cl.build_orders(df_copy)
df_copy[df_copy["Bill Number"] == "G0470211"]

<a id="step-3-7"></a>
#### Step 3.7 - Data limitations
Before analysing, it's important to know what this data **can't** tell us. Some limitations were clear from the start, others we found in later steps (shown in brackets). All conclusions in this notebook should be read with them in mind.

| Limitation | Why it matters | How we handle it |
|---|---|---|
| **No customer ID and no party size** | We can't see repeat customers or how many people were at a table. A bill is not a customer, and "orders per day" counts bills, not people | We talk about orders (bills) and order value, never about customers |
| **Alcohol is billed separately** (Step 6.6) | `N` bills contain only alcohol and `G` bills never do, so a beer and a hookah in the same visit end up on **two bills** (9% of all bills are `N` bills) | To decide in Part 2, e.g. analyse the main bills and the alcohol bills separately |
| **No cost data** | We can't calculate profit or margins, so "best product" always means most revenue, not most profit | We only make claims about revenue |
| **Revenue includes tax**, and tax rates differ | Tobacco and liquor are taxed ~32%, food and drinks ~24%, so the share of tobacco is a bit higher with tax (44.5%) than without (43.1%) | We use the total paid by the customer (incl. tax), as the till records it. The rankings don't change |
| **One café, one year** | We can't separate real growth from a yearly cycle, and we don't know if this café is typical for the chain | Growth and seasons are described for this year only (Step 5) |
| **Prices change during the year** (Step 5.1) | The same products cost ~11% more in 2011, so revenue over time mixes "more sales" and "higher prices" | We split revenue growth into orders, items per order and price per item |
| **We see sales, not the menu** (Step 6.2) | We don't know when a product was on the menu, or if it ran out of stock. A product that was on the menu but never sold is invisible | The time between a product's first and last sale is our estimate of when it was on sale |
| **Repeated rows** (Step 3.3) | 680 rows are exact copies (same bill, item and second). They can be real (the waiter entered the item several times) or a till error | We assume they are real and merge them (Step 3.6). They are only **0.46% of the revenue** (~151k), so this choice doesn't change any conclusion |
| **Names and categories cleaned by us** (Steps 3.4-3.5) | Some fixes are judgement calls, e.g. moving the milkshakes from FOOD to BEVERAGE | Every fix is listed with the reason, so it can be checked or undone |
| **Hot/cold drinks labelled by keywords** (Step 5.3.3) | Labels come from words in the product name, so a few drinks may be in the wrong group | Only used for seasonal trends, where a few mistakes don't change the picture |

<a id="step-4"></a>
### Step 4 - Descriptive Statistics

Here we summarise the data with numbers before looking at trends: the headline figures of the business, the centre and spread of the main variables, the shape of their distributions and the best-selling products.

We look at the data on three levels:
- **Line** (`df_copy`): one item in one order
- **Order** (`df_orders`): one bill, i.e. what a customer spends in one visit
- **Day**: the total revenue of each day

<a id="step-4-1"></a>
#### Step 4.1 - Headline figures

In [ ]:
daily = ts.daily_revenue(df_copy)   # total revenue per day
n_days = len(daily)
ds.overview(df_copy, df_orders, daily)

In one year the café made **32,805,895.27 moneys** in total, from 69,982 orders, which is around 90 thousand moneys per day.

<a id="step-4-2"></a>
#### Step 4.2 - Measures of center and spread

For each variable we calculate:
- **Centre**: mean, median and mode (left empty when no value appears twice, like daily revenue, because then the mode is just a random value)
- **Spread**: standard deviation, range and IQR (Q3 - Q1)
- **Skewness**: how lopsided the distribution is (0 = symmetric, > 0 = long tail to the right)

In [ ]:
ds.stats_table(df_copy, df_orders, daily)

Observations:
- **Every variable is right-skewed** (mean > median, positive skewness): most values are small, with a long tail of large ones.
- **A typical order** has 2 items and costs around 390 moneys (median). The mean (~470) is pulled up by a few very large bills, so the median is the better "typical" value here.
- **Most lines have quantity 1** (Q1 = Q3 = 1), people order more items rather than more of the same item.
- **Daily revenue** is the most stable variable (mean and median are close), but the max day (~250k) is almost 3x the median, worth checking which day that was in Step 5.
- The **min Rate of 0.01** flagged earlier shows up again. We checked it in Step 6: it's a free Mother's Day item, not an error.

<a id="step-4-3"></a>
#### Step 4.3 - Distribution shapes
Histograms show where most of the values are and boxplots show the outliers. The order total histogram is cut at the 99th percentile so the few very large bills don't squash the rest of the plot (the boxplot still shows them).

In [ ]:
ds.plot_distributions(df_orders, daily)
ds.outlier_counts(df_orders, daily)

The plots confirm the skewness found above: order totals and items per order pile up on the left with a long tail of large orders, shown as many outlier points in the boxplots. These large orders are most likely groups and are real sales, so we keep them, but it's another reason to prefer the median over the mean when describing a typical customer.

<a id="step-4-4"></a>
#### Step 4.4 - Most sold products

In [ ]:
ds.plot_top_products(df_copy)

Hookahs are the café's main business: **NIRVANA HOOKAH SINGLE** is number one both in quantity and in revenue. **CAPPUCCINO** is the 2nd most sold item but, being cheap, it doesn't reach the top 5 in revenue, while **SAMBUCA** sells less but is the 2nd biggest earner. Which products drive volume and which drive money will be explored further in Step 6.

<a id="step-4-5"></a>
#### Step 4.5 - Most sold categories

In [ ]:
# Units sold and total revenue per category, plotted side by side
category_sales = ds.category_units(df_copy)
category_revenue = ds.category_revenue(df_copy)
ds.plot_category_sales(category_sales, category_revenue)
ds.category_table(category_sales, category_revenue)

The ranking by revenue is very different from the ranking by units:
- **Tobacco (hookah) brings in the most money**: 14.6M, **44.5%** of the revenue, while it's only 3rd in units sold.
- **Beverage** is 1st in units but 2nd in revenue (25.1%): drinks are cheap. **Food** follows closely with 22.0%.
- **Liquor** adds 6.4%, and the other four categories together make less than 2%.

Step 4.6 looks closer at the top 3, and Step 6.1 at why units and revenue rank so differently.

<a id="step-4-6"></a>
#### Step 4.6 - The top 3 categories
Step 4.5 showed that a few categories make most of the sales. Here we check which ones are the top 3 by units and by revenue, and how much of the business they make up together. The other 5 categories are grouped as "Other".

In [ ]:
top3 = ds.top_categories(category_sales, category_revenue)
top3

In [ ]:
top3_shares = ds.top_category_shares(df_copy, category_sales.index[:3])
ds.plot_top_category_shares(top3_shares)
ds.show_top_category_shares(top3_shares)

- **The top 3 are the same by units and by revenue**: beverage, food and tobacco. Only the order changes.
- **Together they are the café**: **92% of the units, 92% of the revenue**, and at least one of them is in **91% of the orders**.
- **Each one has a different role**:
  - **Beverage brings the volume**: 41% of units and in 53% of orders, but only 25% of revenue (cheap items).
  - **Tobacco brings the money**: only 23% of units, but **44.5% of revenue** and in almost half of the orders (48%).
  - **Food is in between**: 28% of units, 22% of revenue, and the widest menu (194 products).
- **"Other"** (liquor, wines, misc, merchandise, liquor & tobacco) makes only 8% of units and revenue, but has 195 products, as many as the whole food menu. Liquor is almost only beer, and liquor & tobacco is 4 beer + hookah combo deals.

Step 6.1 looks at the revenue per unit of every category.

<a id="step-4-7"></a>
#### Step 4.7 - Top products in the top 3 categories
For each of the top 3 categories, the 5 best products by units sold (left) and by revenue (right). The title shows how much of the category those 5 products make up.

In [ ]:
ds.plot_top_products_in_categories(df_copy, category_sales.index[:3])

- **BEVERAGE**: **Cappuccino** sells the most, but the **Great Lakes Shake** (moved here from FOOD in Step 3) brings in the most money. By revenue the cheap drinks (water, chai, mint tea) drop out and the more expensive **shakes and Red Bull** come in.
- **FOOD**: the same 5 products by units and by revenue (poutine, Jr. Chill Avalanche and the paninis), only **Jr. Chill Avalanche** moves to first place by revenue. Food is spread out: the top 5 make only ~30% of the category.
- **TOBACCO**: very concentrated, the **top 5 make 68%** of the category both ways. **Nirvana Hookah Single** leads both, and by revenue the **N R G hookah** replaces the cheaper Green Apple flavour. Sambuca and Calcutta Mint here are hookah flavours, not drinks.

<a id="step-5"></a>
### Step 5 - Time Series Analysis and Sales Trend Investigation
With a full year of data (Apr 2010 - Mar 2011) we look at how sales change over time: the **trend**, a **decomposition** into trend, seasonality and noise ($Y_t = T_t + S_t + \varepsilon_t$), the **recurring cycles** (hourly, weekly, monthly) and whether the series is **stationary**.

All calculations live in [time_series.py](scripts/time_series.py), so here we only show the results and what they mean.

In [ ]:
df_copy = ts.add_hour(df_copy)
df_orders = ts.add_hour(df_orders)
beverages = ts.label_drinks(df_copy)   # beverages labelled hot / cold / other

<a id="step-5-1"></a>
#### Step 5.1 - Trend isolation

In [ ]:
with ps.grid(True):   # time series: no grid
    ts.plot_monthly_revenue(daily)

Monthly revenue is lowest in **May-June** (~2.4M), rises in July-August, dips in September and then climbs to a **peak in December** (~3.5M, holidays and New Year's Eve). It falls back from January but stays above the spring 2010 level (~2.8M in March vs ~2.4M in April). Months have a different number of days, so below we look at revenue **per day** and smooth out the day-to-day noise.

A **moving average** smooths out the day-to-day noise: 7 days cancels the weekly cycle, 30 days shows the long-term direction.

In [ ]:
ma = ts.moving_averages(daily)
with ps.grid(False):   # time series: no grid
    ts.plot_trend(ma)
ts.trend_summary(daily, ma)

- **Revenue grew over the year**: an average day in Jan-Mar 2011 brought in ~97k, against ~79k in Apr-Jun 2010 (**+24%**). Below we check whether that is more customers or customers paying more.
- The **low point** was mid-May (~74k per day), the **high point** the last week of December (~116k), driven by the holidays and New Year's Eve.
- After December sales go down again, but stay well above the spring 2010 level.
- Caveat: with only one year we can't tell real growth apart from a yearly cycle. A second year of data would settle this.

**Where does the growth come from?** Revenue can grow in three ways: **more orders** (more visits), **more items in each order**, or a **higher price per item**. We split the growth into these three parts to see which one drove it.

In [ ]:
ts.growth_breakdown(df_copy, df_orders)

In [ ]:
with ps.grid(True): # Change for background grid.
    ts.plot_growth_drivers(df_copy, df_orders)

A higher price per item can mean two things: the café **raised its prices**, or customers **chose more expensive products**. To separate them we compare the price of the **same products** in both quarters, weighting each product by how much it sold in Apr-Jun 2010 (a *Laspeyres price index*):

In [ ]:
ts.same_product_price_change(df_copy)

- **Only a third of the +24% comes from more orders** (+8% per day). The rest comes from each order being worth more: **+5% items per order** and **+9% price per item**.
- **The same products cost ~11% more** in Jan-Mar 2011 than in Apr-Jun 2010 (100 of the 265 products sold in both quarters changed price). So the higher price per item is **price increases**, not customers choosing more expensive products.
- **Two phases**: until December growth came from **more orders** (busy summer and holidays). From October the **average order value climbs every month** (+15% by March), while orders per day fall back after December. In **March 2011 there were fewer orders per day than in April 2010**.
- Business implication: revenue is held up by higher prices, but **visits are not growing anymore**. From this data alone we can't tell whether the price increases caused the drop in orders, so we look at the price changes per product in Part 3.

<a id="step-5-2"></a>
#### Step 5.2 - Structural decomposition
Daily revenue split into **trend** (7-day moving average), **weekly seasonality** (average effect of each weekday, $S_{t+7} = S_t$) and **noise** (the rest).

In [ ]:
dec = ts.decompose(daily, period=7)
with ps.grid(False):   # time series: no grid
    ts.plot_decomposition(dec)
ts.variance_share(dec)

In [ ]:
ts.biggest_surprises(dec)

- Three forces of similar size: the **trend explains ~39%** of the daily variation, the **weekly cycle ~24%** and **~34% is noise**.
- **Weekly pattern**: Saturday is ~18k above the trend, Sunday ~6k and Friday ~4k, while Monday to Thursday are 6-9k below it.
- The **noise roughly doubles in the second half of the year**: busier months are less predictable, so a multiplicative model (or the log of revenue) may fit better than an additive one.
- **The biggest surprises are special days and big bookings** (checked in Step 6):
  - **New Year's Eve** made ~250k, almost 3x a normal day. The dip on 1 January is partly a side effect of that huge value on the moving average.
  - **Tue 25 Jan** was ~37k above expected: that day had a **party booking for 23 people** (14,231, the biggest bill of the year).
  - **Valentine's Day** was ~27k above a normal Monday.
  - **Sun 20 Feb** had one of the five biggest group orders (~8k). Sun 24 Oct has no obvious explanation.

<a id="step-5-3"></a>
#### Step 5.3 - Seasonality and recurring cycles
<a id="step-5-3-1"></a>
##### Step 5.3.1 - Hourly cycle

In [ ]:
with ps.grid(True):   # time series: no grid
    ts.plot_hourly(df_copy, df_orders, beverages, n_days)
ts.hourly_summary(df_orders, n_days)

**There is no morning coffee rush, this is an evening and night business:**
- The café opens around **10h** and takes its last orders around **02h**.
- Only **16%** of orders come before 15h, while **54%** come from 19h until closing. The **busiest hour is 19h**, and it stays busy until midnight, apart from a short dip at 21h after the dinner rush.
- In the late morning most drinks are hot (~70% at 10h), but the volume is tiny. From the early afternoon on, about **60% of the drinks are cold** (shakes, iced teas, lemonades).
- **Liquor** peaks around 22h and **tobacco** (hookah) around 23h: the late evening is the bar/lounge part of the business.

<a id="step-5-3-2"></a>
##### Step 5.3.2 - Weekly cycle

In [ ]:
ts.plot_weekly(daily, df_orders)
ts.weekday_table(daily, df_orders, dec)

The heatmap shows that most of Saturday's extra orders come **between 00h and 02h, which is really Friday night**. So we also look at the **business day**, where orders before 06h count for the previous evening.

In [ ]:
business_daily = ts.business_day_revenue(df_orders, daily)
ts.calendar_vs_business_day(daily, business_daily)

- **Weekends are the peak**: by calendar date Saturday makes ~109k against ~81k on Tuesday (**+34%**), with more orders (224 vs 178) and slightly bigger orders.
- By **business day**, **Friday and Saturday nights are equally strong (~99k each)**, followed by Sunday (~95k). Monday to Thursday stay around 81-86k.
- Sunday has the **highest average order value (~501)**, which suggests bigger groups or longer visits.
- The real peaks for staffing are **Friday and Saturday evenings, including the hours after midnight**.

<a id="step-5-3-3"></a>
##### Step 5.3.3 - Monthly / seasonal cycle
Beverages are labelled hot or cold from keywords in their names.
- **Left**: how many hot and cold drinks were sold each month. These numbers also go up and down with how busy the month was.
- **Right**: out of every 100 hot or cold drinks sold that month, how many were **cold**. This doesn't depend on how busy the month was, so it shows what customers **chose**. Above the dashed 50% line more cold drinks were sold, below it more hot drinks.

In [ ]:
drinks = ts.drinks_per_month(beverages)
with ps.grid(False):   # time series: no grid
    ts.plot_drinks_per_month(drinks)

**Hot drinks drive the seasonality, not cold ones:**
- **Cold drinks are stable** all year (~2,400-3,000 per month, including the shakes): no real "summer surge".
- **Hot drinks more than double**, from ~1,230 in May to ~2,880 in December, and stay high in Dec-Jan.
- Cold drinks are the majority in every month, but their share is highest in **April-May (66-70%)** and lowest in **December-January (49-51%)**. Hot drinks also rise in July-August, which fits the cooler monsoon season if the café is in India.
- The slow months (Apr-Jun) are the ones where hot drinks sell least, so promoting cold drinks then could help fill the gap.

<a id="step-5-4"></a>
#### Step 5.4 - Detrending and stationarity
A **stationary** series keeps the same mean and spread over time, which most forecasting models need. We remove the trend by **differencing**: the change vs the day before ($Y'_t = Y_t - Y_{t-1}$) and vs the same weekday last week ($Y_t - Y_{t-7}$). We also try the weekly difference of **log revenue**, and check every series with the **Augmented Dickey-Fuller (ADF) test** (H0: the series is not stationary, so a p-value below 0.05 means stationary).

In [ ]:
diffs = ts.differenced_series(daily)
with ps.grid(False):   # time series: no grid
    ts.plot_differencing(diffs)
ts.stationarity_table(diffs)

- The **original series is not stationary**: the average day rises from ~83k to ~97k between the two halves of the year, the spread grows too, and the ADF test can't reject non-stationarity (p = 0.11).
- **Differencing removes the trend**: all differenced series have a mean close to 0 and pass the ADF test (p < 0.01).
- Only the **weekly differences also remove the weekly cycle** (the first difference still has a lag-7 autocorrelation of 0.26).
- The **spread still grows** with plain differencing (the std of the weekly difference is 1.8x higher in the second half). Taking the **log first** reduces this to 1.2x, so for modelling we recommend **log revenue + weekly differencing**.

<a id="step-5-5"></a>
#### Step 5.5 - Key takeaways
1. **Revenue grew, but mostly through prices**: +24% per day from Apr-Jun 2010 to Jan-Mar 2011. Only +8% comes from more orders, the rest from bigger and more expensive orders (the same products cost ~11% more). Orders per day peaked in December and by March were below April 2010.
2. **It's an evening/night café**: more than half of the orders come after 19h and there is no morning rush. The quiet late morning and lunch hours are the biggest room for growth.
3. **Friday and Saturday nights are the strongest**, about +19% above Monday-Wednesday, with many sales after midnight.
4. **Hot drinks drive the seasons**: they more than double from May to December while cold drinks stay flat.
5. **Special days matter**: New Year's Eve was almost 3x a normal day, and party bookings, group orders and Valentine's Day also stand out.
6. **For forecasting**: use log revenue with weekly differencing (stationary by the ADF test) and add calendar features (weekday, special days, business day) in Step 7.

<a id="step-6"></a>
### Step 6 - Category Performance and Relationship Discovery
Now we look at **what** the café sells: how each category performs, how concentrated sales are, how the mix changes through the day, which products sell best per month and season, how variables relate to each other and which anomalies we found.

In [ ]:
df_copy = cat.add_daypart(df_copy)
df_orders = cat.add_daypart(df_orders)

<a id="step-6-1"></a>
#### Step 6.1 - Category performance
Steps 4.5 and 4.6 showed how much each category sells and earns. Here we look at **why** units and revenue rank so differently, with two numbers per category:
- **Revenue per unit**: the price level of the category
- **Revenue per product per month**: how much an item on the menu earns in a month on sale. Products are on sale for very different lengths of time (a New Year's Eve special for 1 day, a regular product all year, see Step 6.2), so we divide by the total time the products were on sale instead of by the number of products.

In [ ]:
performance = cat.category_performance(df_copy, df_orders)
cat.plot_category_performance(performance)
cat.show_category_performance(performance)

- **Price explains the gap between units and revenue**: a hookah brings in **~389 per unit**, 3x a drink (~123) and 2.5x a food item (~157). That's why tobacco makes 45% of the revenue with 23% of the units.
- **Tobacco's menu works the hardest**: **~36k per product per month**, 2x liquor (~18k), 5x beverage (~7k) and 6.5x food (~5.6k).
- **Liquor is the surprise**: only 33 products (mostly beer), but each earns ~18k a month, more than twice a drink or a dish.
- **Food has the longest menu (194 products)** and earns the least per product of the top 3: a long menu with many small sellers.
- **Wines and merchandise are expensive per unit, but earn little per product** (~1.9k and ~0.3k a month).
- **Liquor & tobacco** has the highest price per unit (~676) because its 4 products are beer + hookah combo deals.

<a id="step-6-2"></a>
#### Step 6.2 - Concentration of sales (Pareto)
The 80/20 rule says a small part of the products often makes most of the revenue. With 577 different products sold during the year, how concentrated is the café's revenue?

In [ ]:
with ps.grid(False):
    cat.plot_pareto(df_copy)
cat.pareto_summary(df_copy)

- Revenue is **very concentrated**: **17 products (3%) make half of the revenue** and **66 products (11%) make 80%**, even stronger than the 80/20 rule.
- That leaves a long tail of products that sell very little. But can we simply cut them?

**Which products really sell too little?**

Counting units over the whole year is misleading, because **not every product was on sale all year**: some were sold on one evening only (New Year's Eve specials), some only arrived with the new menu on 7 January 2011, and some were dropped before it. A product sold for 1 day can't reach 12 units in a year, even if it sold 10 that evening.

We only see sales, not the menu, so we use the time between a product's **first and last sale** as the time it was on sale, and measure **units per day on sale**. Every product gets a lifecycle status:
- **Special**: all its sales fall within one week
- **New**: first sold from 7 January 2011 (new menu)
- **Retired**: last sold before 7 January 2011
- **Regular**: on sale before and after the new menu started

The last column counts the products that sold **fewer than 12 units in the whole year**. That looks like "less than once a month", but it ignores how long each product was on sale:

In [ ]:
lifecycle = cat.product_lifecycle(df_copy)
cat.lifecycle_summary(lifecycle)

Only **36 of the 191 products with fewer than 12 units are regular products**. The others are specials (82), retired (59) or new (14), so their low yearly total says nothing about how popular they are.

So we only look at the products **still on the menu in 2011** (regular and new), and count those that sell less than once a week or once a month **per day on sale**:

In [ ]:
cat.slow_sellers(lifecycle)

In [ ]:
# The slowest food and drinks still on the menu
cat.slowest_products(lifecycle)

- **The real tail is much smaller than it looked**: of the 313 products on the 2011 menu, **78 sell less than once a week** and **23 less than once a month**. Together the 78 bring in **0.9% of the revenue**.
- **Half of them are merchandise**: all 39 merchandise items (T-shirts, notebooks, mugs, ...) sell less than once a week. They don't take kitchen time, so the cost of keeping them is mostly stock.
- **On the kitchen and bar side, the tail is speciality coffee**: 18 drinks sell less than once a week, and **15 of them are speciality coffees**: 8 origins and blends (*Java Estates*, *New Orleans Blue*, *Hawaiian Kona*, *Guatemala Antigua*, ...), most of them both as regular and au lait. Food has only 4 slow products.
- Business implication: there's **no need to cut a third of the menu**. A small clean-up is enough: **group the speciality coffees** into fewer options, review the **10 slow wines**, and keep merchandise as a small corner. Before removing a product we would still check in Part 2 if it's bought together with best sellers.
- Limitation: a product that was on the menu but never sold is invisible in this data, and a product that only sold now and then over a long period counts as "on sale" the whole time.

<a id="step-6-3"></a>
#### Step 6.3 - Category mix through the day and the week
Step 5 showed *when* the café is busy. Here we look at *what* it sells at those moments. We split the day into parts based on the opening hours.

In [ ]:
cat.plot_category_mix(df_copy)
cat.daypart_table(df_copy, df_orders)

- **Hookah is the biggest earner in every part of the day** (39-47% of revenue), even at lunch. It's not only a night product.
- **Drinks matter most in the late morning** (33% of revenue), **alcohol grows through the day**: liquor goes from 1% in the late morning to 9% at night, wine from 0.1% to 2.3%.
- The **evening (19-23h) brings in the most revenue (35%)**, followed by the afternoon (30%) and the night (22%). The late morning is only 2%.
- **Orders get bigger as the day goes on**: the average order is ~360 in the late morning and ~520 at night.
- The **mix is almost the same every weekday**. From Friday to Sunday people order a bit more food and liquor (and drinks on Sat-Sun), so tobacco's share drops from ~45-48% to ~42-43%.

**The top 7 products of the top 3 categories through the day and the week**

The same mix as above, but for the **7 best-selling products** of each of the top 3 categories (Step 4.6). Each bar shows how the units of those 7 products are split, so we can see which of them gain or lose popularity during the day and the week.

In [ ]:
cat.plot_top_product_mix(df_copy)

- **The time of day matters much more than the weekday**: the bars by weekday are almost the same every day, while the bars by part of the day clearly shift. The only weekday effect is on **Sunday**: a bit more Sambuca (18% vs ~15%), B.M.T. panini (21% vs ~17%) and masala chai (13% vs ~10%).
- **BEVERAGE**: **cappuccino is a daytime drink**, 34% of these drinks in the late morning and 21% at night, and caffe latte follows the same pattern (12% -> 7%). **Moroccan mint tea** goes the other way, from 5% to 15% at night, and water is most popular at night (18%). Lemon iced tea is rare in the late morning (5%) and takes ~10% from lunch on.
- **FOOD**: the **Phillycream cheese & chilly panini is the morning favourite** (23% in the late morning, 10% at night), while **Jr. Chill Avalanche** grows from 10% to 27% at night and **poutine** from 16% to 25% in the evening. The B.M.T. panini is stable all day (~18%).
- **TOBACCO**: the cheaper **Nirvana** (33% -> 27%), **Mint flavour** (26% -> 18-22%) and **Silver Apple** (11% -> 6%) are strongest in the late morning and at lunch. From the afternoon the premium **Sambuca** (10% -> 18%) and **N R G** (2% -> 9%) take over part of their share.

<a id="step-6-4"></a>
#### Step 6.4 - Most sold products per month and season
Seasons are the four quarters of the data year.

<a id="step-6-4-1"></a>
##### Step 6.4.1 - Top products per month

In [ ]:
cat.plot_top_products_per_month(df_copy, n=4)

In [ ]:
cat.top_products_per_period(df_copy, df_copy["Date"].dt.to_period("M"))

<a id="step-6-4-2"></a>
##### Step 6.4.2 - Top products per season

In [ ]:
cat.plot_top_products_per_season(df_copy)

In [ ]:
cat.top_products_per_period(df_copy, cat.season_of(df_copy["Date"]), n=5)

<a id="step-6-4-3"></a>
##### Step 6.4.3 - Products gaining or losing share between seasons
The top products barely change, so we also look at which products **gain or lose share** between the warm quarter (Apr-Jun) and the cooler half of the year (Oct-Mar), for products sold in every season:

In [ ]:
season_change = cat.season_changes(df_copy)
cat.plot_season_changes(season_change)
season_change

- The **top products are the same all year**: **Nirvana Hookah Single** is number 1 in 10 of the 12 months (the Great Lakes Shake leads in April 2010, Cappuccino in January 2011), followed by Cappuccino, Great Lakes Shake and Mint Flavour.
- The **Great Lakes Shake** is in the top 3 in most months, but it is the product that **loses the most share** in Oct-Mar compared to Apr-Jun, together with other cold products (Jr. Chill Avalanche, Berry Blast, Miami Melons, Country Lemonade).
- **Winter winners** are hot and "heavier" products: **masala chai**, poutine, and beer (**Tuborg**, KF pitcher).
- **KF Draught 1 litre loses share while the KF pitcher gains**: a possible effect of a price or menu change, a good candidate for Part 3.

<a id="step-6-5"></a>
#### Step 6.5 - Relationships
- **Price vs units sold**: do cheaper products sell more? Units are measured **per day on sale** (Step 6.2) and the one-week specials are left out, so new and short-lived products are compared fairly. We use the **Spearman** correlation because prices and units are very skewed (it compares ranks instead of values).
- **Categories over time**: do the categories rise and fall together from day to day?

<a id="step-6-5-1"></a>
##### Step 6.5.1 - Price vs units sold

In [ ]:
products = cat.price_vs_units(df_copy)
cat.plot_price_vs_units(products)
cat.plot_price_vs_units_by_category(products)
cat.price_units_correlation(products)

<a id="step-6-5-2"></a>
##### Step 6.5.2 - Categories over time

In [ ]:
cat.plot_daily_category_correlation(df_copy)

- **Overall, cheaper products sell a bit more** (Spearman rho = -0.21), but that's mostly because categories have different price levels.
- **Within categories** the picture is different: for **liquor and wine** cheaper clearly sells more (rho -0.74 and -0.75), for **food** price doesn't matter (no significant relation), and for **tobacco the more expensive hookahs sell more** (rho +0.51): customers are willing to pay for premium hookahs.
- **Food, beverage and tobacco rise and fall together** from day to day (correlations 0.54-0.66): they are driven by the same thing, the number of visitors. **Liquor** follows less (0.27-0.38) and **wine** is almost independent (rare, irregular sales).

<a id="step-6-6"></a>
#### Step 6.6 - Anomalies
Things in the data that don't behave like normal sales, found during the previous steps.

<a id="step-6-6-1"></a>
##### Step 6.6.1 - Items sold for 0.01
Flagged in Step 3:

In [ ]:
cat.free_items(df_copy)

<a id="step-6-6-2"></a>
##### Step 6.6.2 - The biggest orders
Flagged as outliers in Step 4:

In [ ]:
cat.biggest_orders(df_copy, df_orders)

<a id="step-6-6-3"></a>
##### Step 6.6.3 - Two bill series
Bill numbers start with `G` or `N`. We compared both:

In [ ]:
cat.bill_series(df_copy, df_orders)

<a id="step-6-6-4"></a>
##### Step 6.6.4 - Menu changes
While fixing the item names (Step 3) we saw names that stop being used in December and new names from January. We count, per month, how many products are sold for the first and the last time:

In [ ]:
print("Days with the most new products in January 2011:")
print(cat.first_sale_dates(df_copy).to_string())
cat.menu_changes(df_copy)

<a id="step-6-6-5"></a>
##### Step 6.6.5 - Summary of the anomalies
| Anomaly | What we found | What we do with it |
|---|---|---|
| **Items at 0.01** | 5 lines of `MOTHERS DAY SPL` on 9 May 2010: a free gift added to bills | Keep: real event, no effect on revenue |
| **Biggest bill (14,231)** | `PARTY CHARGES @ 500/-` x 23 on 25 Jan 2011: a party booking. This explains the 25 Jan surprise in Step 5 | Keep, flag as special day (Step 7) |
| **Very large orders** (40-64 items) | Group orders on 11 Jun, Christmas Day, 2 Jan and 20 Feb | Keep: real sales, part of the order outliers in Step 4 |
| **Two bill series** | `N` bills (9% of orders) contain **only alcohol**, `G` bills **never** contain alcohol: the café bills alcohol separately | Important for Part 2: a beer and a hookah in the same visit end up on **two different bills** |
| **New menu in January 2011** | 37 new products on 7-8 January, 73 new products in January, while 125 products are sold for the last time in Dec-Jan | Flag old/new menu (Step 7), candidate for Part 3 |
| **December MISC spike** | New December products (sheeshas, KF draft mug, spirits) were booked as MISC | Fixed in Step 3 |
| **New Year's Eve** | ~250k, almost 3x a normal day (Step 5) | Flag as special day (Step 7) |

<a id="step-6-7"></a>
#### Step 6.7 - Key takeaways
1. **Hookah is the core business**: 45% of revenue from 23% of the units, the biggest earner in every part of the day, and premium hookahs sell better than cheap ones.
2. **Drinks bring people in, hookah brings in the money**: beverages are 41% of units but only 25% of revenue.
3. **Revenue is concentrated, but the menu tail is smaller than it looks**: 66 products make 80% of the revenue. Most products with low yearly sales were specials, new or retired. Of the 313 products on the 2011 menu, 78 sell less than once a week (0.9% of revenue): merchandise, slow wines and speciality-coffee variants that could be grouped.
4. **Alcohol is billed separately**, so baskets with alcohol and hookah/food are split over two bills. We must keep this in mind in Part 2.
5. **A new menu started on 7 January 2011**, and several big events (party booking, group orders, New Year's Eve) explain the biggest revenue surprises.

<a id="step-7"></a>
### Step 7 - Offensive Data Preparation & Feature Engineering
We turn what we learned in Steps 3-6 into new columns, so the next parts (market basket analysis and our own question) can use them directly. Every feature comes from a finding:

In [ ]:
fe.FEATURES

<a id="step-7-1"></a>
#### Step 7.1 - Promotion and base product
When we searched for duplicate item names in Step 3, many names looked almost identical but had a prefix or suffix like `1+1`, `B1G1` or `2+1`, e.g. `VLN CAB SAUV (GLS)` vs `1+1 VLN CAB SAUV (GLS)`, or `CARLSBERG` vs `CARLSBERG 2+1`. These are the **same base product sold as a deal**, at a different price.

We don't merge them with the normal product, because that would mix different prices and quantities. But we don't want to lose that information either: the dataset has a `Discount` column, and **promotions are not recorded there**. They only show up in the product name. So we add two columns:
- `Promo`: the type of deal, found from patterns in the name
- `Base product`: the name without the deal (`1+1 VLN CAB SAUV (GLS)` -> `VLN CAB SAUV (GLS)`), so a product can be compared with and without its deal

| Promo | Pattern in the name | Examples |
|---|---|---|
| **Buy 1 get 1** | starts with `1+1` or `B1G1` | `1+1 KF 1/2 LITER`, `B1G1 ZINZI RED (GLS)` |
| **Multi-buy** | `2+1`, `3+2`, ... | `CARLSBERG 2+1`, `RED BULL 3+2` |
| **Multi-pack** | starts with 2-9 followed by the product | `3 RED BULL`, `2 AXE TWIST` |
| **Combo** | two products joined by ` + ` | `WAFFLES + BEVERAGE`, `2 DOM BEER + 1SPL SHEESHA` |

Names that start with a number but are not a deal are left out on purpose: `4 SEASONS ...` and `2 OCEAN PINOTAGE` are wine brands, `3COURSE ... MEAL` is a set menu and `1 AXE TWIST` is a single item.

In [ ]:
df_copy = fe.add_line_features(df_copy)
fe.promo_summary(df_copy)

In [ ]:
# Which products are on promotion, and what is their base product?
fe.promo_products(df_copy)

What the promo column tells us:
- **Promotions are a small part of the business**: 42 products on a deal, ~743k in revenue, only **~2.3% of total revenue**.
- **The `Discount` column doesn't capture them**: only 6 of the ~270 discounted lines are promo items. Without this column, promotions would be invisible in the analysis.
- **Red Bull deals are the most used promotion by far**: `RED BULL 2+1` alone sold 1,225 units. It was used until early January 2011, and from February `3 RED BULL` (and `2`/`5 RED BULL`) appear instead. This looks like the same Red Bull deal under a new name and price (250 vs 300), probably part of the new menu.
- **Beer and wine deals were short campaigns**: the Kingfisher `1+1` deals only ran from 12 to 26 September 2010, which explains the spike of promo sales in September. The wine `1+1` deals ran in June-July.
- **Base product**: most deals simply add a prefix or suffix to the regular name. A few beer and Red Bull deals use a short name (`KF 1/2 LITER`, `RED BULL`), so we map those to their regular product (`KF DRAUGHT (1/2LTR)`, `RED BULL ENERGY DRINK`). Deals without a regular version (e.g. `1+1 WINE GLASS`) keep their own name.
- In Part 3 this column lets us compare the same product with and without its deal.

<a id="step-7-2"></a>
#### Step 7.2 - Time, drink and bill features
- `Business day`, `Weekday`, `Weekend`: orders before 06h count for the previous evening, so Friday night stays Friday night (Step 5)
- `Special day`: the holidays and events that stood out in Step 5 and 6 (New Year's Eve, Valentine's Day, the party booking, ...)
- `Menu`: old menu / new menu from 7 January 2011 (Step 6)
- `Drink type`: hot / cold / other drink, or not a drink (Step 5)
- `Bill series`: main bill or alcohol bill (Step 6)

In [ ]:
feature_cols = ["Date", "Time", "Item Desc", "Category", "Daypart", "Business day", "Weekday", "Weekend",
                "Special day", "Menu", "Drink type", "Bill series", "Promo", "Base product"]
df_copy[feature_cols].sample(8, random_state=1)

<a id="step-7-3"></a>
#### Step 7.3 - Order-level features
For the market basket analysis (Part 2) we need one row per order with its size, value, timing and which categories it contains.

In [ ]:
order_features = fe.build_order_features(df_copy)
order_features.head()

In [ ]:
# Share of orders containing each category, and orders with more than one product
has_cols = [c for c in order_features if c.startswith("Has ")]
summary = order_features[has_cols].mean().map("{:.1%}".format).to_frame("Share of orders")
summary.loc["More than one product"] = f"{(order_features['Products'] > 1).mean():.1%}"
summary

- **54% of the orders contain more than one product**, so about 38,000 orders can be used for the market basket analysis in Part 2.
- **Beverage (53%) and tobacco (48%) are the most common** in an order, food is in 38%.
- **Only 13% of the alcohol bills have more than one product** (58% of the main bills), because alcohol is billed on its own. In Part 2 we have to decide how to handle this, e.g. analyse the main bills and the alcohol bills separately.
- Promotions are in only 2.9% of the orders.

<a id="step-7-4"></a>
#### Step 7.4 - Saving the cleaned data
We save the result of Steps 3-7 in the `data/` folder, so the next parts can start from the cleaned data instead of the raw file:
- `cafe_clean_lines.csv`: one row per item in an order, with all fixes from Step 3 and the new columns from Step 7
- `cafe_clean_orders.csv`: one row per order (bill), with its size, value, timing and which categories it contains: the starting point for the market basket analysis (Part 2)

In [ ]:
df_copy.to_csv("data/cafe_clean_lines.csv", index=False)
order_features.to_csv("data/cafe_clean_orders.csv")   # the index is the bill number
print(f"cafe_clean_lines.csv:  {df_copy.shape[0]:,} rows x {df_copy.shape[1]} columns")
print(f"cafe_clean_orders.csv: {order_features.shape[0]:,} rows x {order_features.shape[1] + 1} columns")

<a id="part-2"></a>
## Part 2 - Market Basket Analysis
Which products are bought together, and which pairs would make good **combo deals**? We look for product pairs that appear on the same bill more often than chance, for the whole day and for each part of the day.

The calculations live in [apriori_market_basket_analysis_daypart_overview.py](scripts/apriori_market_basket_analysis_daypart_overview.py) (imported as `mba`), so here we only show the results and what they mean.

<a id="p2-step-1"></a>
### Step 1 - Dataset Selection and Feature Engineering for MBA
Part 1 saved two cleaned datasets (Step 7.4). Both cover the same 69,982 bills, but at a different level:

| Criterion | `cafe_clean_lines.csv` (selected) | `cafe_clean_orders.csv` (not selected) |
|---|---|---|
| **Granularity** | Line-item level (one row per item in a bill) | Order level (one row per bill) |
| **Product detail** | Keeps the product names (`Item Desc`, `Base product`), prices (`Rate`) and discounts | Only bill totals (`Products`, `Units`, `Order_value`) and category flags (`Has FOOD`, ...) |
| **Suitability for combo deals** | High: shows exactly which products are on the same bill | Low: only shows which *categories* are on the same bill |

The features from Part 1 are made for the EDA, so we add four transformations for the basket analysis:
1. **Group on `Base product`** (Step 7.1), so a product and its deal version count as the same product.
2. **Binary presence per bill**: a product counts once per bill (1/0), so ordering 3 of the same item doesn't distort the pair probabilities.
3. **Remove noise**: `ADD ON S` and `MERCHANDISE` are not things anyone would bundle.
4. **Split by `Daypart`** (Step 6.3), because what people order changes a lot through the day.

In [ ]:
mba_lines = mba.load_mba_lines("data/cafe_clean_lines.csv")   # removes ADD ON S and MERCHANDISE
products_per_bill = mba_lines.groupby("Bill Number")["Base product"].nunique()
print(f"Lines used for MBA: {len(mba_lines):,}")
print(f"Bills:              {mba_lines['Bill Number'].nunique():,}")
print(f"Bills with only one product: {(products_per_bill == 1).mean():.1%}")

<a id="p2-step-2"></a>
### Step 2 - Algorithm Selection and Metrics
We use the **Apriori** algorithm. It uses the **Apriori principle**: if a single product is not frequent enough (below `min_support`), no pair containing it can be frequent either, so those pairs are pruned early. The steps are: frequent single products -> candidate pairs -> association rules.

- **Why Apriori**: the steps are transparent and easy to explain, and with ~70k bills it is fast enough to tune the thresholds by hand.
- **Alternatives**: *FP-Growth* builds a prefix tree and skips candidate generation, which is faster on very large data with very low thresholds, but harder to follow step by step. *ECLAT* intersects lists of bill IDs and works best on dense data, not on small café bills.

For a pair of products A (the **anchor**, the product people come for) and B (the **add-on**):

| Metric | Formula | Meaning |
|---|---|---|
| **Support (A, B)** | Bills with A and B / all bills | How common the pair is. Filters out rare coincidences |
| **Confidence (A -> B)** | Bills with A and B / bills with A | Out of the bills with A, how many also have B. The direction matters |
| **Lift (A, B)** | Support(A, B) / (Support(A) x Support(B)) | How much more often they are bought together than by chance. > 1 positive, = 1 independent, < 1 negative |

In the code the product that sells most of the two is the anchor and the other is the add-on.

<a id="p2-step-3"></a>
### Step 3 - Threshold Calibration and Overall Rules
The menu has hundreds of products and almost half of the bills contain only one product, so the support of any pair is naturally low. With `min_support = 1%` no pair is left, and with very low thresholds we get noise. We use:
- **Whole year**: `min_support = 0.3%` (at least 25 bills) and `min_lift = 1.15`
- **Per daypart**: `min_support = 0.4%` (at least 10 bills) and `min_lift = 1.15`

In [ ]:
mba_overall = mba.run_overall(mba_lines)
print(f"Rules for the whole year: {len(mba_overall)}")
mba_overall[["anchor_product", "addon_product", "cat_a", "cat_b", "bills_together", "support_pct",
             "conf_a_to_b_pct", "conf_b_to_a_pct", "lift", "sum_rate"]]

- **Red Bull + Sambuca is the strongest pair of the year**: they are on 612 bills together (support 0.88%). Of the bills with a Red Bull, 18.2% also have a Sambuca hookah (and 13.8% the other way around), giving a **lift of 2.86**: almost 3x more often than by chance.
- **Red Bull goes with hookah**: the top 3 pairs are all Red Bull with a hookah flavour (Sambuca, Calcutta Mint, N R G).
- The **B.M.T. panini** pairs with many products (poutine, water, hookah, Red Bull, shakes): it's the most "shareable" food item.
- All rules have a support below 1%: product pairs are rare, so even the best pairs are a small part of all bills.

<a id="p2-step-4"></a>
### Step 4 - Combo Deals per Daypart
One combo for the whole day would not fit, because what people order changes through the day (Step 6.3). We run Apriori on each daypart separately and rank the pairs by lift: **rank 1 is the best deal** of that daypart, the next two are runners-up.

In [ ]:
mba_daypart = mba.run_by_daypart(mba_lines)
print(f"Rules per daypart: {len(mba_daypart)}")
print(mba_daypart.groupby("daypart", sort=False).size().to_string())
mba.top_per_daypart(mba_daypart, n=3)

- **Late morning (10-12h)**: a **breakfast bundle** of eggs or an omelette with a 1L bottle of mineral water. The best pair by lift (hash brown omelette) is based on only 10 bills, so scrambled eggs + water (19 bills, lift 4.67) is the safer choice.
- **Lunch (12-15h)**: **B.M.T. panini + fries** has the highest lift of all dayparts (6.58): a meal upgrade.
- **Afternoon (15-19h) and evening (19-23h)**: **Red Bull + Sambuca hookah** is the best deal, with the most bills of all pairs (191 and 209). Red Bull with other hookah flavours (Calcutta Mint, N R G) are the runners-up.
- **Night (23-03h)**: **Nirvana hookah + a small bottle of water** (lift 3.25), a late-night hydration bundle. Red Bull + Sambuca is still among the top pairs.

<a id="p2-step-5"></a>
### Step 5 - Exporting the rules
The full rule tables are saved in the `outputs/` folder, so they can be opened outside the notebook.

In [ ]:
for path in mba.save_results(mba_overall, mba_daypart, out_dir="outputs"):
    print(path)

<a id="p2-step-6"></a>
### Step 6 - Recommendations and Limitations
**Recommended combo deal per daypart:**

| Daypart | Combo deal | Idea |
|---|---|---|
| Late morning (10-12h) | Scrambled eggs + Qua mineral water (1L) | Breakfast bundle |
| Lunch (12-15h) | B.M.T. panini + fries | Lunch meal upgrade |
| Afternoon (15-19h) | Red Bull + Sambuca hookah | Afternoon lounge bundle |
| Evening (19-23h) | Red Bull + Sambuca hookah | Peak evening hookah + energy |
| Night (23-03h) | Nirvana hookah single + Qua mineral water (500ml) | Late-night hydration bundle |

**Limitations:**
- **No cost data**: we know the selling price, not the margin, so the discount of a bundle must be checked against the costs first.
- **Correlation is not causation**: a high lift means the products are on the same bill (often a table of several people), not that a discount on A will make people buy B.
- **Alcohol is on separate bills** (Part 1, Step 6.6): pairs of alcohol with food or hookah can't be found in this data.
- **Old data**: April 2010 - March 2011. The menu and prices must be checked before launching a deal.

<a id="part-3"></a>
## Part 3 - Red Bull Promotions, Traffic & Price Controls, and Deal-Buying Orders
In Part 3 we focus on one targeted question using only `cafe_clean_lines.csv` from Part 1: **Do Red Bull promotions actually increase daily Red Bull revenue after accounting for how busy the café is (weekends) and the January 2011 price increase, and what kind of customer order is most likely to buy a deal?**

<a id="p3-step-1"></a>
### Step 1 - Setup & Analytical Questions

<a id="p3-step-1-1"></a>
#### Step 1.1 - Questions and hypotheses
In **Part 1**, we saw that Red Bull items account for the largest share of promotional volume (`RED BULL 2+1` through early January 2011, shifting to `2`, `3`, and `5 RED BULL` multi-packs in February–March 2011), while café traffic peaks on **weekends** and prices rose in **January 2011** (base `RED BULL ENERGY DRINK` moving from `125` to `150` on the new menu from 7 January 2011).

Part 3 investigates two connected questions using `cafe_clean_lines.csv`:
1. **Do Red Bull promotions actually increase daily Red Bull revenue after accounting for how busy the café is (weekends) and the January 2011 price increase?**
2. **What kind of customer order is most likely to buy a deal?**

<a id="p3-step-1-2"></a>
#### Step 1.2 - Libraries and cleaned line data
We load only `cafe_clean_lines.csv` produced at the end of Part 1 (Step 7.4) and derive daily and order-level tables directly from those item lines.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

lines = pd.read_csv("data/cafe_clean_lines.csv")
lines["Date"] = pd.to_datetime(lines["Date"])
lines["Business day"] = pd.to_datetime(lines["Business day"])

print(f"Lines loaded: {lines.shape[0]:,} rows x {lines.shape[1]} columns")

<a id="p3-step-2"></a>
### Step 2 - Red Bull Revenue After Controlling for Weekends and the January 2011 Price Increase

<a id="p3-step-2-1"></a>
#### Step 2.1 - Building the daily Red Bull and café traffic dataset from lines
From `cafe_clean_lines.csv`, we group `Base product == "RED BULL ENERGY DRINK"` lines by `Business day` to get daily Red Bull revenue and units, and group all lines by `Business day` to count unique bills (`day_orders`) and total daily revenue (`day_revenue`), along with `Weekend` and `post_jan1` (`Business day >= 2011-01-01`).

In [ ]:
rb_lines = lines[lines["Base product"] == "RED BULL ENERGY DRINK"].copy()
rb_lines["is_rb_promo"] = rb_lines["Promo"] != "No promo"

day_traffic = (
    lines.groupby("Business day")
    .agg(
        Weekend=("Weekend", "first"),
        Weekday=("Weekday", "first"),
        Menu=("Menu", "first"),
        Special_day=("Special day", "first"),
        day_orders=("Bill Number", "nunique"),
        day_revenue=("Total", "sum"),
    )
    .reset_index()
)

rb_daily = (
    rb_lines.groupby("Business day")
    .agg(
        rb_revenue=("Total", "sum"),
        rb_units=("Quantity", "sum"),
        rb_promo_revenue=("Total", lambda s: s[rb_lines.loc[s.index, "is_rb_promo"]].sum()),
        rb_promo_units=("Quantity", lambda s: s[rb_lines.loc[s.index, "is_rb_promo"]].sum()),
        rb_promo_lines=("Quantity", lambda s: int(rb_lines.loc[s.index, "is_rb_promo"].sum())),
    )
    .reset_index()
)

rb_panel = (
    day_traffic.merge(rb_daily, on="Business day", how="left")
    .fillna({
        "rb_revenue": 0.0,
        "rb_units": 0.0,
        "rb_promo_revenue": 0.0,
        "rb_promo_units": 0.0,
        "rb_promo_lines": 0,
    })
    .sort_values("Business day")
    .reset_index(drop=True)
)

rb_panel["has_rb_promo"] = (rb_panel["rb_promo_lines"] > 0).astype(int)
rb_panel["weekend_int"] = rb_panel["Weekend"].astype(int)
rb_panel["post_jan1"] = (rb_panel["Business day"] >= pd.Timestamp("2011-01-01")).astype(int)
rb_panel["non_special"] = (rb_panel["Special_day"] == "No").astype(int)

rb_panel[["Business day", "Weekday", "Weekend", "day_orders", "has_rb_promo", "rb_units", "rb_revenue", "post_jan1"]].head(10)

<a id="p3-step-2-2"></a>
#### Step 2.2 - Promo vs. non-promo revenue across weekend traffic and January 2011 pricing
We compare **Red Bull promo days vs. non-promo days** inside each **January 2011 period × Weekend/Weekday** bucket to see whether active promotions lift daily Red Bull revenue after accounting for weekend café traffic and post-January 2011 prices.

In [ ]:
cell_summary = (
    rb_panel.groupby(["post_jan1", "weekend_int", "has_rb_promo"])
    .agg(
        days=("Business day", "count"),
        mean_rb_rev=("rb_revenue", "mean"),
        median_rb_rev=("rb_revenue", "median"),
        mean_rb_units=("rb_units", "mean"),
        mean_day_orders=("day_orders", "mean"),
    )
    .round(2)
)
print(cell_summary.to_string())

rb_panel["regime_label"] = (
    np.where(rb_panel["post_jan1"] == 1, "Post-Jan 2011", "Pre-Jan 2011")
    + " | "
    + np.where(rb_panel["weekend_int"] == 1, "Weekend", "Weekday")
)
regime_order = [
    "Pre-Jan 2011 | Weekday",
    "Pre-Jan 2011 | Weekend",
    "Post-Jan 2011 | Weekday",
    "Post-Jan 2011 | Weekend",
]

fig, ax = plt.subplots(figsize=(10.5, 5.2))
sns.barplot(
    data=rb_panel,
    x="regime_label",
    y="rb_revenue",
    hue="has_rb_promo",
    order=regime_order,
    errorbar="ci",
    palette=["#8c96a5", "#1f77b4"],
    ax=ax,
)
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, ["No Red Bull Promo", "Red Bull Promo Active"], title="Promotion Status")
ax.set_title("Mean Daily Red Bull Revenue: Promo vs. No Promo Across Traffic & Jan 2011 Pricing")
ax.set_xlabel("Price Period & Day Type")
ax.set_ylabel("Daily Red Bull Revenue (95% CI)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

<a id="p3-step-2-3"></a>
#### Step 2.3 - Regression check on Red Bull promotion lift
We fit OLS models estimating daily Red Bull revenue (`rb_revenue`) on `has_rb_promo` while controlling for `weekend_int`, `day_orders`, `post_jan1`, and `non_special`, including interaction terms to test whether promo lift differs on weekends or after January 1, 2011.

In [ ]:
m_add = smf.ols(
    "rb_revenue ~ has_rb_promo + weekend_int + post_jan1 + day_orders + non_special",
    data=rb_panel,
).fit()
print("=== Additive Model ===")
print(m_add.summary().tables[1])
print(f"R2: {m_add.rsquared:.4f} | Adj R2: {m_add.rsquared_adj:.4f}\n")

m_int = smf.ols(
    "rb_revenue ~ has_rb_promo * weekend_int + has_rb_promo * post_jan1 + day_orders + non_special",
    data=rb_panel,
).fit()
print("=== Interaction Model (Promo x Weekend, Promo x Post-Jan 2011) ===")
print(m_int.summary().tables[1])
print(f"R2: {m_int.rsquared:.4f} | Adj R2: {m_int.rsquared_adj:.4f}")

<a id="p3-step-3"></a>
### Step 3 - Which Customer Orders Are Most Likely to Buy a Deal?

<a id="p3-step-3-1"></a>
#### Step 3.1 - Aggregating bill-level orders from cafe_clean_lines.csv
Because we are using only `cafe_clean_lines.csv`, we roll up item rows by `Bill Number` to inspect order composition, order size, order value, and deal attachment (`has_any_promo` and `has_rb_promo`).

In [ ]:
lines_work = lines.copy()
lines_work["is_promo"] = lines_work["Promo"] != "No promo"
lines_work["is_rb_promo"] = (lines_work["Base product"] == "RED BULL ENERGY DRINK") & lines_work["is_promo"]

orders_from_lines = (
    lines_work.groupby("Bill Number")
    .agg(
        Business_day=("Business day", "first"),
        Weekday=("Weekday", "first"),
        Weekend=("Weekend", "first"),
        Daypart=("Daypart", "first"),
        Hour=("Hour", "first"),
        Special_day=("Special day", "first"),
        Menu=("Menu", "first"),
        Bill_series=("Bill series", "first"),
        Products=("Item Desc", "nunique"),
        Units=("Quantity", "sum"),
        Order_value=("Total", "sum"),
        Has_FOOD=("Category", lambda s: (s == "FOOD").any()),
        Has_BEVERAGE=("Category", lambda s: (s == "BEVERAGE").any()),
        Has_TOBACCO=("Category", lambda s: (s == "TOBACCO").any()),
        Has_LIQUOR=("Category", lambda s: (s == "LIQUOR").any()),
        Has_WINES=("Category", lambda s: (s == "WINES").any()),
        Has_promo=("is_promo", "any"),
        Has_rb_promo=("is_rb_promo", "any"),
    )
    .reset_index()
)

print(f"Orders aggregated from lines: {len(orders_from_lines):,}")

<a id="p3-step-3-2"></a>
#### Step 3.2 - Deal attachment by order composition, basket size, daypart, and order value
We compare deal rates (`Has_promo` and `Has_rb_promo`) across **category basket mix**, **order item count**, **daypart**, and **order value quartiles**.

In [ ]:
ord_df = orders_from_lines.copy()
ord_df["items_bin"] = pd.cut(
    ord_df["Products"],
    bins=[0, 1, 2, 3, 5, 100],
    labels=["1 item", "2 items", "3 items", "4-5 items", "6+ items"],
)
ord_df["value_quartile"] = pd.qcut(
    ord_df["Order_value"], q=4, labels=["Q1 (Lowest)", "Q2", "Q3", "Q4 (Highest)"]
)
ord_df["basket_mix"] = np.where(
    ord_df["Has_FOOD"] & ord_df["Has_BEVERAGE"] & ord_df["Has_TOBACCO"], "Food + Drink + Hookah",
    np.where(
        ord_df["Has_FOOD"] & ord_df["Has_BEVERAGE"], "Food + Drink",
        np.where(
            ord_df["Has_BEVERAGE"] & ord_df["Has_TOBACCO"], "Drink + Hookah",
            np.where(
                ord_df["Has_FOOD"] & ord_df["Has_TOBACCO"], "Food + Hookah",
                np.where(ord_df["Products"] == 1, "Single-item order", "Other multi-item"),
            ),
        ),
    ),
)

mix_summary = (
    ord_df.groupby("basket_mix", observed=False)
    .agg(
        orders=("Bill Number", "count"),
        any_deal_pct=("Has_promo", lambda s: s.mean() * 100),
        rb_deal_pct=("Has_rb_promo", lambda s: s.mean() * 100),
        avg_order_value=("Order_value", "mean"),
    )
    .sort_values("any_deal_pct", ascending=True)
)
print(mix_summary.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))

axes[0].barh(mix_summary.index, mix_summary["any_deal_pct"], color="#2b5c8f")
for idx, (_, row) in enumerate(mix_summary.iterrows()):
    axes[0].text(
        row["any_deal_pct"] + 0.25,
        idx,
        f"{row['any_deal_pct']:.1f}% (n={int(row['orders']):,})",
        va="center",
        fontsize=8.5,
    )
axes[0].set_xlim(0, mix_summary["any_deal_pct"].max() * 1.28)
axes[0].set_xlabel("Share of Orders with a Deal (%)")
axes[0].set_title("Deal Rate by Customer Order Composition")

items_summary = ord_df.groupby("items_bin", observed=False)["Has_promo"].mean() * 100
axes[1].plot(items_summary.index.astype(str), items_summary.values, marker="o", linewidth=2, color="#d95f02")
for x_i, val in enumerate(items_summary.values):
    axes[1].text(x_i, val + 0.25, f"{val:.1f}%", ha="center", fontsize=8.5)
axes[1].set_ylim(0, items_summary.max() * 1.25)
axes[1].set_xlabel("Distinct Products in Order")
axes[1].set_ylabel("Share of Orders with a Deal (%)")
axes[1].set_title("Deal Likelihood Across Order Sizes")

plt.tight_layout()
plt.show()

<a id="p3-step-3-3"></a>
#### Step 3.3 - Modeling deal likelihood across customer orders
We fit a logistic regression model across all orders (`Has_promo`) controlling for `Products`, `Order_value`, `weekend_int`, `post_jan1`, and category flags (`Has_FOOD`, `Has_BEVERAGE`, `Has_TOBACCO`, `Has_LIQUOR`, `Has_WINES`).

In [ ]:
logit_df = ord_df.assign(
    has_promo_int=ord_df["Has_promo"].astype(int),
    weekend_int=ord_df["Weekend"].astype(int),
    post_jan1=(ord_df["Business_day"] >= pd.Timestamp("2011-01-01")).astype(int),
    has_food_int=ord_df["Has_FOOD"].astype(int),
    has_bev_int=ord_df["Has_BEVERAGE"].astype(int),
    has_tob_int=ord_df["Has_TOBACCO"].astype(int),
    has_liq_int=ord_df["Has_LIQUOR"].astype(int),
    has_wine_int=ord_df["Has_WINES"].astype(int),
)

logit_model = smf.logit(
    "has_promo_int ~ Products + Order_value + weekend_int + post_jan1 + has_food_int + has_bev_int + has_tob_int + has_liq_int + has_wine_int",
    data=logit_df,
).fit(disp=False)

or_table = pd.DataFrame({
    "coef": logit_model.params,
    "odds_ratio": np.exp(logit_model.params),
    "p_value": logit_model.pvalues,
}).round(4)
print(or_table.to_string())

<a id="p3-step-4"></a>
### Step 4 - Key Takeaways & Limitations

<a id="p3-step-4-1"></a>
#### Step 4.1 - Summary of findings
- **Red Bull promotions add positive daily Red Bull revenue even after controls**: controlling for weekend traffic, daily café order count, special days, and the January 2011 price increase, active Red Bull promotion days remain associated with higher daily Red Bull revenue.
- **Both weekends and January 2011 pricing raise the baseline**: weekends and post-January 2011 pricing shift daily Red Bull revenue upward, so evaluating promotions without those controls mixes traffic/price shifts with promo lift.
- **Multi-category and larger orders buy deals more often**: orders combining **Food + Drink + Hookah** or **Drink + Hookah**, larger multi-item baskets, and higher-value (`Q4`) orders show markedly higher deal attachment than single-item orders.

<a id="p3-step-4-2"></a>
#### Step 4.2 - Data limitations
| Limitation | Why it matters | How we handle it |
|---|---|---|
| **Observational timing and promo format shift** | `RED BULL 2+1` / `3+2` ran on the old menu and shifted to `2`/`3`/`5 RED BULL` multi-packs in February–March 2011 alongside the January 2011 repricing | We include `post_jan1` and `day_orders` controls and inspect promo vs. non-promo cells directly |
| **Separate alcohol bills (`N` bills)** | Alcohol and food/hookah/Red Bull items in the same visit can appear on separate bills | We aggregate by `Bill Number` from `cafe_clean_lines.csv` and control for category flags |
| **No customer ID or cost data** | Repeat guests and unit margins are not observed in the till log | Results describe bill-level deal behavior and gross revenue (`Total`) only |